# NFGS manuscript experiments

One overview of a complete reconstruction campaign. The default dataset is **archived manuscript summary tables**. Set `RESULT_ROOT` below to a freshly generated campaign directory to view new results. Missing outputs remain visible; archived values are never silently substituted.

In [ ]:
from pathlib import Path
import sys, json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'nfgs_solver.py').exists())
sys.path.insert(0, str(ROOT))
from nfgs.analysis import *
RESULT_ROOT = ROOT / 'reference_results'  # change to ROOT / 'results/paper' for a fresh campaign
DATA_KIND = 'Archived manuscript tables' if RESULT_ROOT == ROOT / 'reference_results' else 'Fresh campaign outputs'
print(DATA_KIND, RESULT_ROOT.relative_to(ROOT))
tables, missing = load_tables(RESULT_ROOT)
display(pd.DataFrame({'dataset': list(tables), 'rows': [len(v) for v in tables.values()]}))
print('Missing stages:', missing or 'none')
if (RESULT_ROOT / 'campaign_state.json').exists():
    state = json.loads((RESULT_ROOT / 'campaign_state.json').read_text())
    display(pd.DataFrame([dict(experiment=k, status=v['status'], seconds=v.get('wall_time_s')) for k,v in state['experiments'].items()]))
plt.rcParams.update({'font.size': 12, 'text.color': 'black', 'axes.labelcolor': 'black'})
summary_path=RESULT_ROOT/'evidence_summary.json'
summary=json.loads(summary_path.read_text()) if summary_path.exists() else {}
SELECTED_RHO=float(summary.get('common_choice',{}).get('rho',.5))
print('Common diagnostic choice:',SELECTED_RHO)


## Physical and numerical protocol

The input grids, source parameters, masks, current operator and stopping rule are explicit. A quick profile is a numerical check, not a replacement for the manuscript campaign.

In [ ]:
protocol=json.loads((ROOT/'configs/paper_protocol.json').read_text())
display(protocol)

## CCI fidelity and force-freeness

The ring marks the common weight recorded in evidence_summary.json (0.5 is the fallback for older archives). Every plotted current residual uses the common second-order curl and the primary scoring domain.

In [ ]:
td=primary_td(tables['td']) if 'td' in tables else pd.DataFrame()
if not td.empty:
    plot_tradeoff(td, selected_rho=SELECTED_RHO); plt.show()
    display(td[np.isclose(td.rho,SELECTED_RHO)][['benchmark','rho','E_rel_Bgt_domain','sigma_J']])

## Agreement with the three-dimensional reference field

In [ ]:
if not td.empty:
    plot_field_metrics(td); plt.show()
    display(td[np.isclose(td.rho,SELECTED_RHO)][['benchmark','C_vec','C_CS','Eprime_n','Eprime_m','epsilon']])

## Published NFPT, VFVP and SSW comparison

These are published scores from Yi et al. (2022), not results from a new execution of those solvers. Field agreement improves most consistently for TD1 and TD2; the original current diagnostic retains its published discretization.

In [ ]:
published=pd.read_csv(ROOT/'reference_results/published_methods.csv')
display(published)
if not td.empty:
    selected=td[np.isclose(td.rho,SELECTED_RHO)].merge(published[published.method=='NFPT'],left_on='benchmark',right_on=published[published.method=='NFPT']['case'].str.lower(),suffixes=('_NFGS','_NFPT'))
    metrics=['C_vec','C_CS','Eprime_n','Eprime_m','epsilon']
    for metric in metrics:
        left=selected[metric+'_NFGS']; right=selected[metric+'_NFPT']
        selected[metric+'_better']='NFGS'
        selected.loc[(abs(left-1)>=abs(right-1)) if metric=='epsilon' else (left<=right),metric+'_better']='NFPT'
    display(selected[['benchmark']+[k+'_better' for k in metrics]])

## Runtime

These are measured solver intervals on the recorded hardware, without a hardware-matched claim of speedup over Fortran NFPT.

In [ ]:
if 'runtime' in tables:
    plot_runtime(tables['runtime']); plt.show()

## Independent Low–Lou family at rho = 0.5

The common curl uses separate dx, dy and dz. The three source profiles are retained in both precisions.

In [ ]:
for key in ['low_lou_fp32','low_lou_fp64']:
    if key in tables:
        print(key)
        display(tables[key][['benchmark','C_vec','C_CS','Eprime_n','Eprime_m','epsilon','sigma_J_common_fd2']])
        plot_grouped(tables[key],['C_vec','sigma_J_common_fd2'],key); plt.show()

## Height-dependent residual and boundary contributions

In [ ]:
if 'height' in tables:
    plot_height(tables['height']); plt.show()
if 'budgets' in tables: display(tables['budgets'])

## Field-line connectivity and high-Q overlap

In [ ]:
if 'topology' in tables:
    display(tables['topology'])
    plot_grouped(tables['topology'],['high_q_jaccard','endpoint_pair_rmse','endpoint_boundary_pair_match_fraction'],'Connectivity diagnostics'); plt.show()
if 'Q_threshold' in tables: display(tables['Q_threshold'])

## Twist-core location and amplitude

In [ ]:
if 'twist' in tables:
    display(tables['twist'])
    plot_grouped(tables['twist'],['core_apex_displacement_mm','peak_abs_Tw_relative_error'],'Twist-core diagnostics'); plt.show()

## Six-face energy and current-carrying helicity

In [ ]:
if 'helicity' in tables:
    display(tables['helicity'])
    plot_grouped(tables['helicity'],['free_energy_over_potential','current_carrying_helicity_fraction_abs'],'Six-face energy and helicity'); plt.show()

## Sensitivity of the practical weight choice

In [ ]:
if 'selection' in tables:
    plot_selection(tables['selection']); plt.show()
    display(tables['selection'].loc[tables['selection'].groupby('cci_squared_term_weight').mean_distance.idxmin()])

## Additional numerical experiments

The extended campaign provides the factorized bridge and resolution matrix. These are distinct from the primary manuscript weight scan.

In [ ]:
for key in ['regularization','factorized','resolution']:
    if key in tables:
        print(key); display(tables[key])
if (RESULT_ROOT/'reproduction_audit.json').exists(): display(json.loads((RESULT_ROOT/'reproduction_audit.json').read_text()))

## Field structure and Q maps

Archived views below retain the original manuscript rendering. A fresh campaign produces common-seed field-line and diagnostic views in its own figures directory; the new rendering does not alter the numerical metrics.

In [ ]:
from IPython.display import Image
figure_dir=RESULT_ROOT/'figures'
if figure_dir.exists():
    for file in sorted(figure_dir.glob('*.png')):
        print(file.name); display(Image(filename=str(file),width=1000))
else: print('No field or map views in this campaign yet.')

In [ ]:
baseline_path=RESULT_ROOT/'low_lou_comparison.json'
if baseline_path.exists():
    low=pd.DataFrame(json.loads(baseline_path.read_text()))
    display(low[['benchmark','rho','E_rel_Bgt','E_rel_Bgt_L2','C_vec','Eprime_n','epsilon','sigma_J']])
    fig,axs=plt.subplots(1,3,figsize=(15,4),layout='constrained')
    for ax,key in zip(axs,['E_rel_Bgt','sigma_J','C_vec']):
        for case,g in low.groupby('benchmark'):
            g=g.sort_values('rho');ax.plot(g.rho,g[key],'-o',label=case)
        ax.set(xlabel='rho',ylabel=key);ax.grid(alpha=.2);ax.legend()
    plt.show()
else: print('Low--Lou baseline evidence has not been exported for this campaign.')
